In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv("Heart.csv")
df.head()


df['AHD'] = df['AHD'].map({'Yes': 1, 'No': -1})


class_yes = df[df['AHD'] == 1]
class_no = df[df['AHD'] == -1]

plt.figure(figsize=(8,6))

plt.scatter(class_yes['Age'], class_yes['Chol'],
            color='red', label='Heart Attack (+1)')

plt.scatter(class_no['Age'], class_no['Chol'],
            color='blue', label='No Heart Attack (-1)')

plt.xlabel("Age")
plt.ylabel("Cholesterol")
plt.title("Age vs Cholesterol")
plt.legend()
plt.show()


X = df[['Age', 'Chol']].values
y = df['AHD'].values

# Convert to 0/1 for logistic math
y_binary = np.where(y == -1, 0, 1)


np.random.seed(42)

indices = np.random.permutation(len(X))

split = int(0.75 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y_binary[train_idx]
y_test = y_binary[test_idx]


mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


def add_bias(X):
    return np.c_[np.ones(X.shape[0]), X]

X_train = add_bias(X_train)
X_test = add_bias(X_test)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def train_logistic(X, y, lr=0.05, epochs=1000):
    
    m, n = X.shape
    w = np.zeros(n)
    
    train_errors = []
    test_errors = []
    
    for _ in range(epochs):
        
        z = np.dot(X, w)
        predictions = sigmoid(z)
        
        gradient = np.dot(X.T, (predictions - y)) / m
        w -= lr * gradient
        
       
        train_pred = (predictions >= 0.5).astype(int)
        train_error = np.mean(train_pred != y)
        train_errors.append(train_error)
        
        test_pred = sigmoid(np.dot(X_test, w))
        test_pred_class = (test_pred >= 0.5).astype(int)
        test_error = np.mean(test_pred_class != y_test)
        test_errors.append(test_error)
        
    return w, train_errors, test_errors


weights, train_errors, test_errors = train_logistic(
    X_train, y_train, lr=0.05, epochs=1000)

print("Weights:", weights)


plt.plot(train_errors, label="Training Error")
plt.plot(test_errors, label="Test Error")

plt.xlabel("Epoch")
plt.ylabel("Error")
plt.legend()
plt.show()


X_plot = X_train[:,1:]

plt.scatter(X_plot[y_train==1][:,0],
            X_plot[y_train==1][:,1],
            color='red')

plt.scatter(X_plot[y_train==0][:,0],
            X_plot[y_train==0][:,1],
            color='blue')

x_vals = np.linspace(X_plot[:,0].min(), X_plot[:,0].max(), 100)
y_vals = -(weights[0] + weights[1]*x_vals) / weights[2]

plt.plot(x_vals, y_vals, 'k')
plt.title("Training Data with Separator")
plt.show()




FileNotFoundError: [Errno 2] No such file or directory: 'Heart.csv'